# Assignment5: CRM Cleanup @ **DalaShop**
*Files (Ch.10), Exceptions (Ch.10), Unit Tests (Ch.11), and Regular Expressions*  
.....

**Total: 3 points**  (Two questions, 1.5 pts each)  

> This assignment is scenario-based and aligned with Python Crash Course Ch.10 (files & exceptions), Ch.11 (unit testing with `unittest`), and Regular Expressions.

## Scenario
You are a data intern at an online retailer called **DalaShop**.  
Sales exported a **raw contacts** file from the CRM. It contains customer names, emails, and phone numbers, but the formatting is messy and some emails are invalid.  
Your tasks:

1. **Clean** the contacts (Files + Exceptions + Regex).  
2. **Write unit tests** to make sure your helper functions work correctly and keep working in the future.

## Data file (given by the company): `contacts_raw.txt`
Use this exact sample data (you may extend it for your own testing, but do **not** change it when submitting).  
Run the next cell once to create the file beside your notebook.

In [ ]:
# Create the provided company dataset file
with open("contacts_raw.txt", "w", encoding="utf-8") as f:
    f.write('Alice Johnson <alice@example.com> , +1 (469) 555-1234\nBob Roberts <bob[at]example.com> , 972-555-777\nSara M. , sara@mail.co , 214 555 8888\n"Mehdi A." <mehdi.ay@example.org> , (469)555-9999\nDelaram <delaram@example.io>, +1-972-777-2121\nNima <NIMA@example.io> , 972.777.2121\nduplicate <Alice@Example.com> , 469 555 1234')
print("Wrote contacts_raw.txt with sample DalaShop data.")

## Q1 (1.5 pts) — CRM cleanup with Files, Exceptions, and Regex
Implement `q1_crm_cleanup.py` to:

1. **Read** `contacts_raw.txt` using `pathlib` and `with`. If the file is missing, **handle** it gracefully with `try/except FileNotFoundError` (print a friendly message; do not crash).
2. **Validate emails** with a simple regex (`r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"`).  
   - Trim whitespace with `strip()` before checking.  
   - Use **full** matching (not partial).
3. **Normalize phone numbers:** remove all non-digits (e.g., with `re.sub(r"\D", "", raw)`).  
   - If the result has **≥ 10 digits**, keep the **last 10 digits**.  
   - Otherwise, return an **empty string** (`""`).
4. **Filter rows:** keep **only** rows with a valid email.
5. **Deduplicate:** remove duplicates by **email** using **case-insensitive** comparison (e.g., `email.casefold()`). **Keep the first occurrence** and drop later duplicates.
6. **Output CSV:** write to `contacts_clean.csv` with **columns exactly** `name,email,phone` (UTF-8).  
7. **Preserve input order:** the order of rows in `contacts_clean.csv` must match the **first appearance** order from the input file. **Do not sort** the rows.

**Grading rubric (1.5 pts):**
- (0.4) File read/write via `pathlib` + graceful `FileNotFoundError` handling  
- (0.5) Correct email regex validation + filtering  
- (0.4) Phone normalization + case-insensitive de-dup (keep first)  
- (0.2) Clean code, clear names, minimal docstrings/comments

In [27]:
from pathlib import Path
import csv
import re
from typing import Iterable, Tuple, List

EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")

def is_valid_email(email: str) -> bool:
    email = (email or "").strip()
    return bool(EMAIL_RE.fullmatch(email))

def normalize_phone(raw: str) -> str:
    digits = re.sub(r"\D", "", raw or "")
    return digits[-10:] if len(digits) >= 10 else ""

def parse_rows(lines: Iterable[str]) -> Iterable[Tuple[str, str, str]]:
    for line in lines:
        if not line.strip():
            continue
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 3:
            continue
        name, email, phone = parts[0], parts[1], ",".join(parts[2:])
        yield name, email, phone

def q1_crm_cleanup(
    in_path: Path = Path("contacts_raw.txt"),
    out_path: Path = Path("contacts_clean.csv"),
) -> None:
    try:
        raw_text = in_path.read_text(encoding="utf-8")
    except FileNotFoundError:
        print("File not found. Please add 'contacts_raw.txt' to the folder.")
        return

    seen = set()
    cleaned: List[Tuple[str, str, str]] = []

    for name, email, phone in parse_rows(raw_text.splitlines()):
        if not is_valid_email(email):
            continue
        key = email.casefold()
        if key in seen:
            continue
        seen.add(key)

        phone_clean = normalize_phone(phone)
        cleaned.append((name.strip(), email.strip(), phone_clean))

    with out_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["name", "email", "phone"])
        writer.writerows(cleaned)

    print(f"{len(cleaned)} contacts written -> contacts_clean.csv")

q1_crm_cleanup()





2 contacts written -> contacts_clean.csv


## Q2 (1.5 pts) — Unit testing with `unittest`
Create tests in `test_crm_cleanup.py` that cover at least:

1. **Email validation**: valid/invalid variations.  
2. **Phone normalization**: parentheses, dashes, spaces, country code; too-short cases.  
3. **Parsing**: from a small multi-line string (not from a file), assert the exact structured rows (name/email/phone).  
4. **De-duplication**: demonstrate that a case-variant duplicate email is dropped (first occurrence kept).




In [29]:
import unittest
from __main__ import normalize_phone, is_valid_email, parse_rows

class TestCRMCleanup(unittest.TestCase):

    def test_email_validation(self):
        self.assertTrue(is_valid_email("user@example.com"))
        self.assertTrue(is_valid_email("john.smith@my-domain.co"))
        self.assertFalse(is_valid_email("bademail@"))
        self.assertFalse(is_valid_email("noatsymbol.com"))

    def test_phone_normalization(self):
        self.assertEqual(normalize_phone("(123) 456-7890"), "1234567890")
        self.assertEqual(normalize_phone("+1 234 567 8901"), "2345678901")
        self.assertEqual(normalize_phone("12-34-56"), "")
        self.assertEqual(normalize_phone("1234567890123"), "4567890123")

    def test_parsing_rows(self):
        text = "John,john@mail.com,1234567890\nJane,jane@mail.com,0987654321"
        lines = text.splitlines()
        result = list(parse_rows(lines))
        expected = [
            ("John", "john@mail.com", "1234567890"),
            ("Jane", "jane@mail.com", "0987654321")
        ]
        self.assertEqual(result, expected)

    def test_deduplication_case_insensitive(self):
        lines = [
            "John,john@mail.com,123",
            "John2,JOHN@mail.com,456",
            "Jane,jane@mail.com,789"
        ]
        rows = list(parse_rows(lines))
        seen = set()
        kept_emails = []
        for _, email, _ in rows:
            key = email.lower()
            if key not in seen:
                seen.add(key)
                kept_emails.append(email)
        self.assertEqual(kept_emails, ["john@mail.com", "jane@mail.com"])

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)


....
----------------------------------------------------------------------
Ran 4 tests in 0.002s

OK


## Grading rubric (total 3 pts)
- **Q1 (1.5 pts)**  
  - (0.4) File I/O with `pathlib` + graceful `FileNotFoundError` handling  
  - (0.5) Email validation (regex + strip + full match) and filtering  
  - (0.4) Phone normalization and **case-insensitive** de-duplication (keep first)  
  - (0.2) Code clarity (names, minimal docstrings/comments)
- **Q2 (1.5 pts)**  
  - (0.6) Meaningful coverage for email/phone functions (valid & invalid)  
  - (0.6) Parsing & de-dup tests that assert exact expected rows  
  - (0.3) Standard `unittest` structure and readable test names
